In [1]:
import sys
print(sys.executable)

d:\Github\.venv-delta\Scripts\python.exe


In [2]:
import pyspark
from delta import configure_spark_with_delta_pip

builder = (
    pyspark.sql.SparkSession.builder
    .appName("PhoenixDelta")
    .master("local[*]")
    .config(
        "spark.sql.extensions",
        "io.delta.sql.DeltaSparkSessionExtension"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    .config(
        "spark.sql.warehouse.dir",
        "D:/Github/Stock-Price-Prediction/data/delta_lake"
    )
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [3]:
print(spark.conf.get("spark.sql.warehouse.dir"))

file:/D:/Github/Stock-Price-Prediction/data/delta_lake


In [50]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType
csv_filepath = r'D:/Github/Stock-Price-Prediction/data/csv_raw/nasdaq_screener.csv';
schema = StructType([
    StructField("Symbol", StringType(), False),
    StructField("Name", StringType(), False),
    StructField("LastSale", StringType(), True),
    StructField("NetChange", FloatType(), True),
    StructField("%Change", StringType(), True),
    StructField("MarketCap", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("IPOyear", IntegerType(), True),
    StructField("Volume", FloatType(), True),
    StructField("Sector", StringType(), True),
    StructField("Industry", StringType(), True)
]);
df = spark.read.csv(csv_filepath, header=True, schema=schema)

spark.sql("create database if not exists bronze")
spark.sql("use bronze")
spark.sql("drop table if exists nasdaq_screener_delta")
df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("nasdaq_screener_delta")

In [51]:
df.show()

+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|Symbol|                Name|LastSale|NetChange|%Change|     MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|
+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|     A|Agilent Technolog...| $132.86|      2.6| 1.996%|37523908080.00| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|  $44.35|     0.87| 2.001%|11703515956.00| United States|   2016|  6846054.0|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|  $10.51|     0.01| 0.095%|          0.00| United States|   2025|    28508.0|                NULL|                NULL|
| AACBR|Artius II Acquisi...|   $0.18|      0.0|  0.00%|          0.00| United States|   2025|      130.0|

In [52]:
spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-07-27 20:31:...|  NULL|    NULL|CREATE OR REPLACE...|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 1, n...|        NULL|Apache-Spark/4.0....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

In [53]:
df_silver = spark.sql("select * from nasdaq_screener_delta")
# df_silver = spark.read.format("delta").table("nasdaq_screener_delta")
# df_silver = spark.read.format("delta").load(r"D:\Github\Stock-Price-Prediction\data\delta_lake\bronze.db\nasdaq_screener_delta")

spark.sql("create database if not exists silver")
spark.sql("use silver")
spark.sql("drop table if exists nasdaq_screener_delta")
# df_silver = spark.read.csv(csv_filepath, header=True, schema=schema)
df_silver.write.format("delta").mode("overwrite").saveAsTable("nasdaq_screener_delta")

In [54]:
spark.sql("SELECT * FROM nasdaq_screener_delta").show()

+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|Symbol|                Name|LastSale|NetChange|%Change|     MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|
+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|     A|Agilent Technolog...| $132.86|      2.6| 1.996%|37523908080.00| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|  $44.35|     0.87| 2.001%|11703515956.00| United States|   2016|  6846054.0|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|  $10.51|     0.01| 0.095%|          0.00| United States|   2025|    28508.0|                NULL|                NULL|
| AACBR|Artius II Acquisi...|   $0.18|      0.0|  0.00%|          0.00| United States|   2025|      130.0|

In [55]:
spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      0|2026-07-27 20:32:...|  NULL|    NULL|CREATE OR REPLACE...|{partitionBy -> [...|NULL|    NULL|     NULL|       NULL|  Serializable|        false|{numFiles -> 1, n...|        NULL|Apache-Spark/4.0....|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+-----------

In [56]:
from pyspark.sql import functions as F

df = spark.read.table("nasdaq_screener_delta")

null_counts = []
for c in df.columns:
    null_counts.append(F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls"))

df.select(*null_counts).show()

+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+
|Symbol_nulls|Name_nulls|LastSale_nulls|NetChange_nulls|%Change_nulls|MarketCap_nulls|Country_nulls|IPOyear_nulls|Volume_nulls|Sector_nulls|Industry_nulls|
+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+
|           0|         0|             0|              0|            1|            398|          306|         2991|           0|         746|           747|
+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+



In [57]:
spark.sql("""
UPDATE nasdaq_screener_delta
SET `%Change` = 0 where `%Change` IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET MarketCap = 0 where MarketCap IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET IPOyear = 2026 where IPOyear IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET Volume = 0 where Volume IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET Industry = 'Unknown' where Industry IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET Sector = 'Unknown' where Sector IS NULL
""")

spark.sql("""
UPDATE nasdaq_screener_delta
SET Country = 'Unknown' where Country IS NULL
""")

DataFrame[num_affected_rows: bigint]

In [58]:
from pyspark.sql import functions as F

df = spark.read.table("nasdaq_screener_delta")

null_counts = []
for c in df.columns:
    null_counts.append(F.sum(F.col(c).isNull().cast("int")).alias(c + "_nulls"))

df.select(*null_counts).show()

+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+
|Symbol_nulls|Name_nulls|LastSale_nulls|NetChange_nulls|%Change_nulls|MarketCap_nulls|Country_nulls|IPOyear_nulls|Volume_nulls|Sector_nulls|Industry_nulls|
+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+
|           0|         0|             0|              0|            0|              0|            0|            0|           0|           0|             0|
+------------+----------+--------------+---------------+-------------+---------------+-------------+-------------+------------+------------+--------------+



In [60]:
df_silver = spark.sql("select * from nasdaq_screener_delta")
df_silver.show()

+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|Symbol|                Name|LastSale|NetChange|%Change|     MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|
+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|     A|Agilent Technolog...| $132.86|      2.6| 1.996%|37523908080.00| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|  $44.35|     0.87| 2.001%|11703515956.00| United States|   2016|  6846054.0|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|  $10.51|     0.01| 0.095%|          0.00| United States|   2025|    28508.0|             Unknown|             Unknown|
| AACBR|Artius II Acquisi...|   $0.18|      0.0|  0.00%|          0.00| United States|   2025|      130.0|

In [61]:
spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      6|2026-07-27 20:33:...|  NULL|    NULL|              UPDATE|{predicate -> ["i...|NULL|    NULL|     NULL|          5|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/4.0....|
|      5|2026-07-27 20:33:...|  NULL|    NULL|              UPDATE|{predicate -> ["i...|NULL|    NULL|     NULL|          4|  Serializable|        false|{numRemoved

In [66]:
from pyspark.sql.types import BooleanType
new_schema = StructType([
    StructField("Symbol", StringType(), False),
    StructField("Name", StringType(), False),
    StructField("LastSale", StringType(), True),
    StructField("NetChange", FloatType(), True),
    StructField("%Change", StringType(), True),
    StructField("MarketCap", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("IPOyear", IntegerType(), True),
    StructField("Volume", FloatType(), True),
    StructField("Sector", StringType(), True),
    StructField("Industry", StringType(), True),
    StructField("IsGoodStock", IntegerType(), True)
]);

In [67]:
new_csv_file = r'D:/Github/Stock-Price-Prediction/data/csv_raw/nasdaq_screener_modified.csv'
df_add = spark.read.csv(new_csv_file, header=True, schema=new_schema)


df_add.show()

+------+--------------------+--------+---------+-------+-----------+--------------+-------+-----------+--------------------+--------------------+-----------+
|Symbol|                Name|LastSale|NetChange|%Change|  MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|IsGoodStock|
+------+--------------------+--------+---------+-------+-----------+--------------+-------+-----------+--------------------+--------------------+-----------+
|     A|Agilent Technolog...| $132.86|      2.6|  2.00%|37523908080| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|          0|
|    AA|Alcoa Corporation...|  $44.35|     0.87|  2.00%|11703515956| United States|   2016|  6846054.0|         Industrials|            Aluminum|          0|
|  AACB|Artius II Acquisi...|  $10.51|     0.01|  0.10%|          0| United States|   2025|    28508.0|                NULL|                NULL|          0|
| AACBR|Artius II Acquisi...|   $0.18|      0.0|  0.

In [68]:
from pyspark.sql.functions import current_timestamp
df2 = df_add.withColumn("update_time", current_timestamp())

In [71]:
spark.sql(""" use bronze """)
df2.write.mode("overwrite").format("delta").option("mergeSchema", "true").saveAsTable("nasdaq_screener_delta")

In [72]:
df_bronze = spark.sql("select * from bronze.nasdaq_screener_delta")
df_bronze.show()

+------+--------------------+--------+---------+-------+-----------+--------------+-------+-----------+--------------------+--------------------+-----------+--------------------+
|Symbol|                Name|LastSale|NetChange|%Change|  MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|IsGoodStock|         update_time|
+------+--------------------+--------+---------+-------+-----------+--------------+-------+-----------+--------------------+--------------------+-----------+--------------------+
|     A|Agilent Technolog...| $132.86|      2.6|  2.00%|37523908080| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|          0|2026-07-27 20:38:...|
|    AA|Alcoa Corporation...|  $44.35|     0.87|  2.00%|11703515956| United States|   2016|  6846054.0|         Industrials|            Aluminum|          0|2026-07-27 20:38:...|
|  AACB|Artius II Acquisi...|  $10.51|     0.01|  0.10%|          0| United States|   2025|    28508.0|  

In [76]:
# First append new data to the existing table and then check the history of the table to see the changes made.
#then overwrite the existing table with the new data and check the history of the table to see the changes made.

spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show(truncate=False)

+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------+----+--------+---------+-----------+--------------+-------------+-----------------------------------------------------------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp              |userId|userName|operation                        |operationParameters                                                                           |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                                                                                 |userMetadata|engineInfo                         |
+-------+-----------------------+------+--------+---------------------------------+----------------------------------------------------------------------------------------------+----+---

In [ ]:
# Time Travel Queries
#use version number or timestamp to query the table as it was at that point in time.
spark.sql("""
SELECT * FROM nasdaq_screener_delta VERSION AS OF 0
""").show()

+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|Symbol|                Name|LastSale|NetChange|%Change|     MarketCap|       Country|IPOyear|     Volume|              Sector|            Industry|
+------+--------------------+--------+---------+-------+--------------+--------------+-------+-----------+--------------------+--------------------+
|     A|Agilent Technolog...| $132.86|      2.6| 1.996%|37523908080.00| United States|   1999|  2383955.0|         Industrials|Biotechnology: La...|
|    AA|Alcoa Corporation...|  $44.35|     0.87| 2.001%|11703515956.00| United States|   2016|  6846054.0|         Industrials|            Aluminum|
|  AACB|Artius II Acquisi...|  $10.51|     0.01| 0.095%|          0.00| United States|   2025|    28508.0|                NULL|                NULL|
| AACBR|Artius II Acquisi...|   $0.18|      0.0|  0.00%|          0.00| United States|   2025|      130.0|

In [80]:
spark.sql("""
SELECT * FROM nasdaq_screener_delta TIMESTAMP AS OF '2026-07-27 20:37:10.427'
where isGoodStock = 1
""").show()

+------+--------------------+----------+---------+-------+-----------+-------------+-------+---------+--------------------+--------------------+-----------+--------------------+
|Symbol|                Name|  LastSale|NetChange|%Change|  MarketCap|      Country|IPOyear|   Volume|              Sector|            Industry|IsGoodStock|         update_time|
+------+--------------------+----------+---------+-------+-----------+-------------+-------+---------+--------------------+--------------------+-----------+--------------------+
|  ASML|ASML Holding N.V....|  $1801.51|    62.49|  3.59%|   6.94E+11|  Netherlands|   1995|1681106.0|          Technology|Industrial Machin...|          1|2026-07-27 20:37:...|
|   AZO|AutoZone Inc. Com...|  $3013.03|     18.9|  0.63%|49188784376|United States|   NULL| 160219.0|Consumer Discreti...|Auto & Home Suppl...|          1|2026-07-27 20:37:...|
| BAC^L|Bank of America C...|  $1281.33|     6.11|  0.48%|       NULL|United States|   NULL|  19875.0|        

In [ ]:
#I deleted accidentally the records with isGoodStock = 0, so I will restore the table to the previous version and then delete the records with isGoodStock = 0 again.
spark.sql("""
DELETE FROM nasdaq_screener_delta WHERE isGoodStock = 0
""").show()

+-----------------+
|num_affected_rows|
+-----------------+
|             7067|
+-----------------+



In [83]:
spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      3|2026-07-27 20:45:...|  NULL|    NULL|              DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemovedFiles ...|        NULL|Apache-Spark/4.0....|
|      2|2026-07-27 20:38:...|  NULL|    NULL|CREATE OR REPLACE...|{partitionBy -> [...|NULL|    NULL|     NULL|          1|  Serializable|        false|{numFiles -

In [84]:
#Restore to previous version
spark.sql("""
RESTORE TABLE nasdaq_screener_delta TO VERSION AS OF 2
""").show()

+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|table_size_after_restore|num_of_files_after_restore|num_removed_files|num_restored_files|removed_files_size|restored_files_size|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+
|                  348110|                         1|                1|                 1|              6506|             348110|
+------------------------+--------------------------+-----------------+------------------+------------------+-------------------+



In [85]:
spark.sql("""
DESCRIBE HISTORY nasdaq_screener_delta
""").show()

+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|version|           timestamp|userId|userName|           operation| operationParameters| job|notebook|clusterId|readVersion|isolationLevel|isBlindAppend|    operationMetrics|userMetadata|          engineInfo|
+-------+--------------------+------+--------+--------------------+--------------------+----+--------+---------+-----------+--------------+-------------+--------------------+------------+--------------------+
|      4|2026-07-27 20:46:...|  NULL|    NULL|             RESTORE|{version -> 2, ti...|NULL|    NULL|     NULL|          3|  Serializable|        false|{numRestoredFiles...|        NULL|Apache-Spark/4.0....|
|      3|2026-07-27 20:45:...|  NULL|    NULL|              DELETE|{predicate -> ["(...|NULL|    NULL|     NULL|          2|  Serializable|        false|{numRemoved

In [ ]:
#some optimizations on gold layer tables
#table.optimize().executeCompaction() -- It will optimize the table by compacting small files into larger files, which can improve query performance. This is especially useful for tables that are frequently updated or have a lot of small files.
#table.optimize().executeZOrderBy("isGoodStock") -- It will optimize the table by reordering the data based on the specified columns, which can improve query performance for certain types of queries. This is especially useful for tables that are frequently queried with filters on those columns.
#table.optimize().vacuum() -- It will remove old versions of the data and free up space, which can improve query performance. This is especially useful for tables that are frequently updated or have a lot of small files.